# 🚀 AI Designer System — Colab Backend

Chạy backend FastAPI trên Google Colab (GPU miễn phí T4) với ngrok tunnel.

**Lưu ý:** Upload thư mục `app/` và file cần thiết lên Google Drive trước, HOẶC dùng code cells bên dưới để clone từ Git.

---

### BƯỚC 1: Clone hoặc upload project

In [ ]:
# Option A: Clone từ Git repo (THAY ĐỔI URL NÀY)
# !git clone https://github.com/YOUR_USERNAME/AI_GEN.git /content/AI_GEN

# Option B: Upload thủ công — chạy cell này rồi dùng Files panel bên trái
# (Kéo thả thư mục app/ vào /content/AI_GEN)

import os
os.makedirs('/content/AI_GEN', exist_ok=True)
os.chdir('/content/AI_GEN')
print('Working directory:', os.getcwd())

---

### BƯỚC 2: Cài đặt dependencies

In [ ]:
# Cài đặt PyTorch với CUDA (T4 = CUDA 12.1)
!pip install torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu121 -q

# Cài đặt các gói cần thiết
!pip install fastapi uvicorn[standard] python-multipart aiofiles pydantic python-dotenv -q
!pip install diffusers transformers accelerate peft safetensors huggingface_hub -q
!pip install opencv-python opencv-contrib-python scikit-image scipy Pillow numpy tqdm -q
!pip install open-clip-torch -q

# Cài ngrok (tunnel public URL)
!pip install pyngrok -q

print('✅ Dependencies installed!')

---

### BƯỚC 3: Thiết lập HuggingFace Token (CÓ GPU MIỄN PHÍ)

In [ ]:
# HuggingFace token — cần thiết để download SDXL Turbo
# Lấy token tại: https://huggingface.co/settings/tokens
# (Token free có sẵn, chỉ cần đăng ký huggingface.co)

import os
import getpass

hf_token = os.environ.get('HF_TOKEN', '')
if not hf_token:
    hf_token = getpass.getpass('Nhập HuggingFace Token (sẽ không hiện khi gõ): ')

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_TOKEN'] = hf_token

print('✅ HuggingFace token đã set')

---

### BƯỚC 4: Thiết lập ngrok (tạo public URL)

In [ ]:
# ĐĂNG KÝ NGROK MIỄN PHÍ: https://ngrok.com/
# Sau khi đăng ký, lấy Authtoken tại: https://dashboard.ngrok.com/get-started/your-authtoken

import getpass

ngrok_token = os.environ.get('NGROK_TOKEN', '')
if not ngrok_token:
    ngrok_token = getpass.getpass('Nhập ngrok Authtoken: ')

from pyngrok import conf
conf.get_default().auth_token = ngrok_token

# Kiểm tra ngrok đã cài
!ngrok config add-authtoken $ngrok_token 2>/dev/null || true
print('✅ ngrok configured')

---

### BƯỚC 5: Tạo file cấu hình .env cho Colab

In [ ]:
# Ghi file .env cho backend (Colab không có .env sẵn)
env_content = f"""
# Inference Backend: cuda để dùng GPU T4 miễn phí
INFERENCE_BACKEND=cuda

# GPU Model — SDXL Turbo (chất lượng cao, hỗ trợ LoRA)
GPU_MODEL=stabilityai/sdxl-turbo

# CPU Model — tiny-sd (fallback)
CPU_MODEL=segmind/tiny-sd

# LoRA auto-load
LORA_AUTO_LOAD=true

# Inference profile
LOCAL_INFER_PROFILE=fast

# Compile on load
COMPILE_ON_LOAD=false

# HuggingFace Token
HF_TOKEN={os.environ.get('HF_TOKEN', '')}
"""

with open('/content/AI_GEN/.env', 'w') as f:
    f.write(env_content.strip())

print('✅ .env created')
print('Contents:')
print(env_content.strip())

---

### BƯỚC 6: Copy LoRA adapters (nếu có train sẵn)

In [ ]:
# Nếu bạn đã train LoRA ở máy local, upload thư mục outputs/ lên Google Drive
# rồi chạy cell này. Nếu không có LoRA, bỏ qua cell này.

# Từ Google Drive (nếu đã mount)
# !cp -r /content/drive/MyDrive/AI_GEN/outputs /content/AI_GEN/outputs

# Hoặc: Clone từ Git repo có sẵn LoRA
# !git clone https://github.com/YOUR_USERNAME/AI_GEN_outputs.git /content/AI_GEN/outputs

print('⚠️  Bước này tùy chọn — bỏ qua nếu chưa có LoRA trained')

---

### BƯỚC 7: Khởi động Backend FastAPI + ngrok tunnel

In [ ]:
import os
import threading
import time
import logging

# Tăng logging để thấy rõ backend chạy
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Chuyển vào thư mục project
os.chdir('/content/AI_GEN')

# Import sau khi đã cd
from pyngrok import ngrok

# Kill cổng 8000 nếu còn process cũ
!pkill -f 'uvicorn' 2>/dev/null || true
time.sleep(1)

# Tạo ngrok tunnel trước khi start server
http_tunnel = ngrok.connect(8000, bind_tls=True)
public_url = http_tunnel.public_url
api_url = public_url + '/api'

print('='*60)
print('🎉 BACKEND PUBLIC URL:')
print(f'   {public_url}')
print(f'   API Base: {api_url}')
print('='*60)
print()
print('📋 Các endpoint:')
print(f'   Health:     {public_url}/health')
print(f'   Generate:   {api_url}/generate')
print(f'   Model:      {api_url}/model/status')
print(f'   LoRA:       {api_url}/lora')
print()
print('⏳ Backend đang khởi động (warmup model)...')
print('   Lần đầu chạy sẽ tải SDXL Turbo (~6GB) — cần 2-5 phút')
print()
# Lưu URL ra file để dễ copy
with open('/content/colab_backend_url.txt', 'w') as f:
    f.write(f'BACKEND_URL={public_url}\n')
    f.write(f'API_BASE={api_url}\n')
print('✅ URL saved to /content/colab_backend_url.txt')

In [ ]:
# Start uvicorn server trong thread riêng
import subprocess
import threading

def run_server():
    import os as _os
    _os.chdir('/content/AI_GEN')
    subprocess.run([
        'python', '-m', 'uvicorn',
        'app.main:app',
        '--host', '0.0.0.0',
        '--port', '8000',
        '--log-level', 'info'
    ], env={
        **os.environ,
        'PYTHONPATH': '/content/AI_GEN',
    })

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Đợi server khởi động
import urllib.request
import time

print('⏳ Đang đợi server khởi động...')
for i in range(30):
    time.sleep(5)
    try:
        resp = urllib.request.urlopen(f'{public_url}/health', timeout=5)
        if resp.status == 200:
            print('✅ Server đã sẵn sàng!')
            break
    except:
        print(f'   Đợi... ({i+1}/30)')
else:
    print('⚠️  Server chưa reply sau 150s — có thể đang load model...')

---

### BƯỚC 8: Kiểm tra backend

In [ ]:
# Test health check
import urllib.request
import json

try:
    resp = urllib.request.urlopen(f'{public_url}/health', timeout=10)
    data = json.loads(resp.read())
    print('📊 Health check:')
    print(json.dumps(data, indent=2))
except Exception as e:
    print(f'⚠️  Health check failed: {e}')
    print('   Backend có thể vẫn đang warmup — đợi thêm và thử lại cell tiếp theo')

In [ ]:
# Test model status
try:
    resp = urllib.request.urlopen(f'{api_url}/model/status', timeout=10)
    data = json.loads(resp.read())
    print('📊 Model status:')
    print(json.dumps(data, indent=2))
except Exception as e:
    print(f'⚠️  {e}')

---

## ✅ HOÀN TẤT!

### Cách sử dụng:

1. **Copy public URL bên trên** (dạng `https://xxxx.ngrok-free.app`)
2. **Cập nhật frontend** — sửa `VITE_API_BASE` trong file cấu hình:
   ```
   VITE_API_BASE=https://xxxx.ngrok-free.app/api
   ```
3. **Build và chạy frontend local** bình thường

### Lưu ý quan trọng:
- **Ngrok free**: tunnel sống ~2 giờ, hết hạn thì restart cell BƯỚC 7
- **GPU T4 free**: 90 phút liên tục, hết thì reconnect Colab
- **Để giữ alive**: Cell BƯỚC 7 phải luôn chạy (đừng stop notebook)
- **Nên dùng Colab Pro** nếu cần runtime dài hơn

In [ ]:
# Quick test generate (tùy chọn)
import urllib.request
import json

test_body = json.dumps({
    'prompt': 'logo tech startup minimalist blue',
    'num_images': 1,
    'mode': 'turbo',
    'width': 512,
    'height': 512
}).encode()

try:
    req = urllib.request.Request(
        f'{api_url}/generate',
        data=test_body,
        headers={'Content-Type': 'application/json'}
    )
    resp = urllib.request.urlopen(req, timeout=60)
    data = json.loads(resp.read())
    print('✅ Generate thành công!')
    print(f'   Ảnh: {len(data.get("images", []))} ảnh')
    print(f'   Metadata: {json.dumps(data.get("metadata", {}), indent=2)}')
except Exception as e:
    print(f'⚠️  Generate failed: {e}')
    print('   Backend có thể vẫn đang warmup lần đầu...')